In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df1:pd.DataFrame=pd.read_csv('idealista18_1.csv')
df2:pd.DataFrame=pd.read_csv('idealista18_2.csv')
df:pd.DataFrame=pd.concat([df1,df2],ignore_index=True).drop_duplicates()

In [ ]:
original:pd.DataFrame=pd.concat([df1,df2],ignore_index=True).drop_duplicates()

In [ ]:
pd.set_option("display.max_columns", None)
df

In [ ]:
unicos:pd.Series=df.nunique(dropna=False)
tipos:pd.Series=df.dtypes
nnans:pd.Series=df.isna().sum().sort_values(ascending=False)
constantes=unicos[unicos==1].keys()
binarios=unicos[unicos==2].keys()
nans=nnans[nnans>0].keys()

In [ ]:
nans

In [ ]:
df=df.drop(columns=constantes)

In [ ]:
df=df.assign(
    BUILTTYPEID=np.select(
        [df['BUILTTYPEID_1']==1,df['BUILTTYPEID_2']==1,df['BUILTTYPEID_3']==1],
        [1,2,3]
    )
)

In [ ]:
df.info()

In [ ]:
df

In [ ]:
df['PERIOD'].value_counts()

In [ ]:
sns.histplot(
    x='PERIOD',
    data=df,
    bins=50,
)
plt.show()

In [ ]:
precios=df['PRICE']
k=1.5
media_precio=precios.mean()
q1=np.quantile(precios,q=0.25)
q3=np.quantile(precios,q=0.75)
iqr=q3-q1
lim_i=q1-iqr*k
lim_s=q3+iqr*k
sns.boxplot(
    x='CITYNAME',
    y='PRICE',
    data=df
)

In [ ]:
plt.figure(figsize=(15,8))
sns.histplot(
    x='PRICE',
    data=df
)
plt.show()

In [ ]:
sns.histplot(
    x='UNITPRICE',
    hue='CITYNAME',
    data=df,
)

In [ ]:
nulos_conteo = df.isnull().sum()
# shape devuelve una tupla (filas, columnas), y con [0] cogemos solo las filas
nulos_porcentaje = (df.isnull().sum() / df.shape[0]) * 100

# Creamos una tabla nueva con las dos cosas juntas para verlo más claro
tabla_nulos = pd.DataFrame({
    "nulos":       nulos_conteo,        # columna con el número de nulos
    "porcentaje":  nulos_porcentaje     # columna con el porcentaje
})

tabla_nulos = tabla_nulos.sort_values("porcentaje", ascending=False)

tabla_nulos[tabla_nulos["nulos"] > 0]

nulos_usuario   = df["CONSTRUCTIONYEAR"].isnull().sum()
nulos_catastro  = df["CADCONSTRUCTIONYEAR"].isnull().sum()
total_filas     = len(df)

print(f"Nulos en CONSTRUCTIONYEAR :    {nulos_usuario}  ({nulos_usuario/total_filas*100:.2f}%)")
print(f"Nulos en CADCONSTRUCTIONYEAR : {nulos_catastro}  ({nulos_catastro/total_filas*100:.2f}%)")
nulos_usuario   = df["CONSTRUCTIONYEAR"].isnull().sum()
nulos_catastro  = df["CADCONSTRUCTIONYEAR"].isnull().sum()
total_filas     = len(df)

print(f"Nulos en CONSTRUCTIONYEAR :    {nulos_usuario}  ({nulos_usuario/total_filas*100:.2f}%)")
print(f"Nulos en CADCONSTRUCTIONYEAR: {nulos_catastro}  ({nulos_catastro/total_filas*100:.2f}%)")
print(f"CONSTRUCTIONYEAR : {df['CONSTRUCTIONYEAR'].describe()}")
print(f"CADCONSTRUCTIONYEAR : {df['CADCONSTRUCTIONYEAR'].describe()}")

mascara = df["CONSTRUCTIONYEAR"].notna() & df["CADCONSTRUCTIONYEAR"].notna()

df_comparacion = df[mascara][["CONSTRUCTIONYEAR", "CADCONSTRUCTIONYEAR"]].copy()

print(f"Filas donde las dos columnas tienen dato: {len(df_comparacion)}")
print(f"Eso es el {len(df_comparacion)/total_filas*100:.1f}% del total de filas")

df_comparacion["diferencia"] = (
    df_comparacion["CONSTRUCTIONYEAR"].astype(int) -
    df_comparacion["CADCONSTRUCTIONYEAR"].astype(int)
)

print("Primeras filas con la diferencia calculada:")
df_comparacion.head(10)


coinciden_exacto = (df_comparacion["diferencia"] == 0).sum()

coinciden_margen_1 = (df_comparacion["diferencia"].abs() <= 1).sum()
coinciden_margen_5 = (df_comparacion["diferencia"].abs() <= 5).sum()

total_comparadas = len(df_comparacion)

print(f"Coinciden exactamente:         {coinciden_exacto}  ({coinciden_exacto/total_comparadas*100:.1f}%)")
print(f"Diferencia de máx. 1 año:      {coinciden_margen_1}  ({coinciden_margen_1/total_comparadas*100:.1f}%)")
print(f"Diferencia de máx. 5 años:     {coinciden_margen_5}  ({coinciden_margen_5/total_comparadas*100:.1f}%)")
print(f"Discrepancia mayor de 5 años:  {total_comparadas - coinciden_margen_5}  ({(total_comparadas - coinciden_margen_5)/total_comparadas*100:.1f}%)")

# Creamos una columna nueva llamada "anio_construccion_final"
# La lógica es:
#   1. Si el catastro tiene dato → usamos ese
#   2. Si el catastro no tiene dato → usamos el del usuario
#   3. Si ninguno tiene dato → queda NaN


df["ANIO_CONSTRUCCION_FINAL"] = df["CADCONSTRUCTIONYEAR"].fillna(df["CONSTRUCTIONYEAR"])


nulos_final = df["ANIO_CONSTRUCCION_FINAL"].isnull().sum()
print(f"Nulos en la columna unificada: {nulos_final}  ({nulos_final/total_filas*100:.2f}%)")

In [ ]:
df=df[(df['DISTANCE_TO_CITY_CENTER']<150)]

In [ ]:
df=df.drop(columns=['CONSTRUCTIONYEAR'])

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df

In [ ]:
correlaciones = df.select_dtypes(include="number").corr()
correlaciones.max().sort_values(ascending=False)
matriz = correlaciones.max().sort_values(ascending=False)
df_matriz = pd.DataFrame(matriz)
(correlaciones-correlaciones.max().sort_values(ascending=False)).max().sort_values(ascending=False)
corr = correlaciones.copy()
valores = corr.to_numpy().copy()
np.fill_diagonal(valores, np.nan)
corr_sin_diag = pd.DataFrame(valores,columns=corr.columns,index=corr.index)


resultado = corr_sin_diag.max().sort_values(ascending=False)
resultado

In [ ]:
# Tanto en general como en Madrid, Barcelona y Valencia por separado
# 1. que es lo que mas determina el precio->el área. En Valencia el precio unitario influye más, lo que significa que los demás aspectos son más significativos que en las demás ciudades
# 2. como se distribuyen los precios->La ciudad más difícil para comprar un piso es Barcelona, aunque tenga un promedio mas bajo que Madrid, debido a que Madrid los precios son muy dispersos. Se sabe por la mediana. La más asequible es Valencia
# 3. media y mediana de precios->Ya se sabe
# 4. como influye el nº de baños en las demás variables->
# 5. tendencia de precio según el período
# 6. que el lo que mas influye en el precio por metro cuadrado
# 8. como son los las diferentes variables de los pisos en promedio

In [ ]:
df.groupby('CITYNAME')['PRICE'].describe()

In [ ]:
df=df[(df['DISTANCE_TO_CITY_CENTER']<150)]
ciudades=df['CITYNAME'].unique()
plt.figure(figsize=(20,20))
for i,c in enumerate(ciudades):
    plt.subplot(3,len(ciudades),(i+1))
    sns.histplot(
        x='PRICE',
        data=df[df['CITYNAME']==c]
    )
    plt.title(c)
    plt.subplot(3,len(ciudades),(i+4))
    sns.histplot(
        x='UNITPRICE',
        data=df[df['CITYNAME']==c]
    )
    plt.subplot(3,len(ciudades),(i+7))
    sns.scatterplot(
        x='CONSTRUCTEDAREA',
        y='PRICE',
        data=df[df['CITYNAME']==c]
    )
plt.show()

In [ ]:
sns.scatterplot(
    x='LATITUDE',
    y='UNITPRICE',
    data=df[(df['CITYNAME']=='Madrid')]
)

In [ ]:
df[df['CITYNAME']=='Madrid'].corr(numeric_only=True)['UNITPRICE'].sort_values(ascending=False).iloc[1:]

In [ ]:
df.corr(numeric_only=True)['PRICE'].sort_values(ascending=False).iloc[1:]